# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

In [1]:
%cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [2]:
%%writefile requirements.txt
google-adk[a2a]==1.19.0
a2a-sdk[http-server]==0.3.16
mcp==1.19.0
agent-framework==1.0.0b251120
agent-framework-a2a==1.0.0b251120
langchain-mcp-adapters
langgraph
langchain[openai]
google-auth==2.42.0
httpx==0.28.1
ipython==8.32.0
nest-asyncio==1.6.0
python-dotenv==1.0.1
requests>=2.32.4,<3.0.0
uvicorn==0.34.0

Overwriting requirements.txt


![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [2]:
import os

from IPython.display import IFrame, Markdown, display

url_template = os.environ.get("DLAI_LOCAL_URL")
url = url_template.format(port=8888) if url_template else None
# Terminal 1: uv run a2a_policy_agent.py
if url:
    IFrame(f"{url}terminals/1", width=800, height=200)
else:
    display(Markdown("Open a local terminal and run `uv run a2a_policy_agent.py`."))

Open a local terminal and run `uv run a2a_policy_agent.py`.

In [19]:
!uv run a2a_policy_agent.py

Running A2A Health Insurance Policy Agent
INFO:     Started server process [79051]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('::1', 9999, 0, 0): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/server.py:68> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/uvicorn/server.py", line 163, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1584, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on a

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [8]:
# Terminal 2: uv run a2a_research_agent.py
if url:
    IFrame(f"{url}terminals/2", width=800, height=200)
else:
    display(Markdown("Open another local terminal and run `uv run a2a_research_agent.py`."))

Open another local terminal and run `uv run a2a_research_agent.py`.

In [3]:
!uv run a2a_research_agent.py

Traceback (most recent call last):
  File "/content/drive/MyDrive/A2A-MCP/a2a_research_agent.py", line 5, in <module>
    from google.adk.a2a.utils.agent_to_a2a import to_a2a
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/google/adk/__init__.py", line 16, in <module>
    from .agents.llm_agent import Agent
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/google/adk/agents/__init__.py", line 49, in <module>
    from .mcp_instruction_provider import McpInstructionProvider
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/google/adk/agents/mcp_instruction_provider.py", line 33, in <module>
    from ..tools.mcp_tool.mcp_session_manager import MCPSessionManager
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/google/adk/tools/mcp_tool/__init__.py", line 23, in <module>
    from .mcp_tool import MCPTool
  File "/content/drive/MyDrive/A2A-MCP/.venv/lib/python3.12/site-packages/google/adk/tools

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [4]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [5]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [6]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [7]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [8]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [9]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [11]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


There are numerous options for getting mental health therapy, and the best approach for you will depend on your individual needs, symptoms, and treatment goals. Therapy is a cornerstone of treatment for many mental health conditions and can help you process feelings, change habits, and grow as a person.

Here's a breakdown of how you can access mental health therapy:

**1. Consult Your Primary Care Provider:**
Your primary care provider can conduct an initial mental health screening and refer you to a mental health professional, such as a social worker, psychologist, or psychiatrist. They can also help you understand which local providers are covered by your insurance plan.

**2. Explore Different Types of Therapy:**
There are many types of therapy, and they often involve communication between a therapist and patient, or therapist and group members. Some common types include:
*   **Cognitive Behavioral Therapy (CBT):** Focuses on identifying, understanding, and changing thinking and behavior patterns. It can be helpful for conditions like depression and anxiety disorders.
*   **Dialectical Behavioral Therapy (DBT):** Integrates cognitive-behavioral techniques with mindfulness to help manage intense emotions and improve interpersonal effectiveness.
*   **Psychotherapy (Talk Therapy):** This is a broad term for discussing mental health issues with a trained professional. It can include various approaches like psychodynamic therapy (exploring past experiences) and interpersonal therapy (focusing on relationships).
*   **Trauma Therapy:** Therapies like EMDR (Eye Movement Desensitization and Reprocessing) and TRM (Trauma Resilience Model) help individuals process past traumas.
*   **Humanistic Therapy:** Focuses on personal growth and self-discovery, supporting individuals in making choices to live their best lives.
*   **Integrative and Holistic Therapy:** Combines techniques from different therapeutic methods to create a personalized plan, often focusing on the whole person—mind, body, and spirit.
*   **Somatic Therapy:** Uses psychotherapy and physical therapies like movement, yoga, or dance to help process emotions.

Prescription Digital Therapeutics (PDTs) are also available, which are FDA-authorized software-based treatments that can be prescribed by healthcare providers.

**3. Utilize Online Directories and Resources:**
Several online platforms can help you find a therapist:
*   **ADAA's Find Your Therapist directory:** Offers a way to locate providers.
*   **Psychology Today's Therapy Directory:** Allows you to search for professionals by zip code, city, or name, and provides details on their approach, specialties, and fees.
*   **GoodTherapy.org:** A directory of licensed therapists and counselors with search filters to help you find a suitable match.
*   **Mental Health Match:** Provides personalized therapist matching based on your needs and identity.
*   **Zencare:** Offers personalized therapist matching and a list of vetted professionals based on specialty, identity, and more.
*   **Headway:** Helps you find therapists covered by your insurance, processing payments to make care more affordable.

**4. Check with Insurance Providers and Employers:**
*   **Insurance Companies:** Contact your health insurance provider for a list of covered therapists. Many insurance companies have online databases or apps for this purpose.
*   **Employee Assistance Programs (EAP):** If your employer offers an EAP, it may provide access to mental health services.

**5. Access Community and Educational Resources:**
*   **Local Mental Health Organizations:** Your local Mental Health America affiliate can provide information about affordable treatment services.
*   **Universities and Teaching Hospitals:** May offer low-cost or free treatment services.
*   **Student Health Centers:** If you are a student, your school's health center may offer mental health services.

**6. For Immediate Help:**
*   **988 Suicide & Crisis Lifeline:** Call or text 988 for 24-hour, confidential support from trained crisis counselors.
*   **Veterans Crisis Line:** Veterans can call 988 and press "1," text 838255, or chat online for confidential support.
*   **Disaster Distress Helpline:** Call or text 1-800-985-5990 for immediate crisis counseling related to disasters.
*   **Emergency Services:** In life-threatening situations, call 911 or go to the nearest emergency room.

When choosing a therapist, consider their experience with your specific concerns, their therapeutic approach, fees, availability, and whether you feel a sense of trust and connection with them. Therapy is often most effective when combined with medication, if appropriate.

If you need mental health, behavioral health, or substance abuse services, you will pay 10% coinsurance for office visits and other outpatient services if you use an In-Network Provider. If you use an Out-of-Network Provider, you will pay 30% coinsurance for office visits and other outpatient services. For inpatient services, you will pay 10% coinsurance as an In-Network Provider and 30% coinsurance as an Out-of-Network Provider.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)